# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_1color

This notebook builds the first scoped submission for the `fill_enclosed_regions / nonlocal_1color` subtype.

Workflow:

1. Load the strict 41-task subtype from `task_type_map.csv`.
2. Try bounded wider additive-template models for tasks where 3x3 context is insufficient.
3. Fall back to compact additive candidates when they visibly fit.
4. Emit identity fallback models for any remaining task so the submission zip is complete.
5. Build `/kaggle/working/submission.zip` from exactly this subtype scope.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-nonlocal-1color-v0.1"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})




def make_initializer(name, array):
    arr = np.asarray(array, dtype=np.float16)
    return numpy_helper.from_array(arr, name=name)

def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-nonlocal-1color-v0.1"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path


In [2]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
SUBTYPE = 'nonlocal_1color'
MODEL_VERSION = 'fill-additive-nonlocal-1color-v0.1'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / f'{FAMILY}_{SUBTYPE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)


DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color
MODEL_VERSION = fill-additive-nonlocal-1color-v0.1


In [3]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.2 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [4]:
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()


def parse_color_list(value):
    if pd.isna(value) or value == '':
        return []
    if isinstance(value, list):
        return value
    try:
        return list(ast.literal_eval(str(value)))
    except Exception:
        return []

family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
nonlocal_1color_df = family_df[
    family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
    & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
    & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
].copy()
nonlocal_1color_df = nonlocal_1color_df.sort_values('task_id').reset_index(drop=True)
task_ids = nonlocal_1color_df['task_id'].tolist()

print('family:', FAMILY)
print('subtype:', SUBTYPE)
print('family tasks:', len(family_df))
print('selected nonlocal_1color tasks:', len(task_ids))
print(task_ids)
display(nonlocal_1color_df.head(10))


family: fill_enclosed_regions
subtype: nonlocal_1color
family tasks: 59
selected nonlocal_1color tasks: 41
['task002', 'task027', 'task042', 'task043', 'task047', 'task050', 'task060', 'task063', 'task090', 'task102', 'task105', 'task119', 'task126', 'task139', 'task162', 'task166', 'task176', 'task200', 'task219', 'task232', 'task246', 'task251', 'task255', 'task265', 'task273', 'task278', 'task299', 'task303', 'task323', 'task335', 'task336', 'task341', 'task348', 'task350', 'task357', 'task367', 'task371', 'task381', 'task387', 'task392', 'task397']


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes,parsed_new_output_colors
0,task002,2,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,5,1,262,268,same_shape_variable_size,...,9828,NaN,0.919284,1421,17605,1.0,9828,9828,Same shape; input is mostly preserved while ne...,[4]
1,task027,27,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,261,265,same_shape_variable_size,...,1260,NaN,0.951500,291,6000,1.0,1260,1260,Same shape; input is mostly preserved while ne...,[2]
2,task042,42,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,1071,NaN,0.948667,308,6000,1.0,1071,1071,Same shape; input is mostly preserved while ne...,[8]
3,task043,43,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,5392,NaN,0.799167,1205,6000,1.0,5392,5392,Same shape; input is mostly preserved while ne...,[2]
4,task047,47,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,2,1,262,265,same_shape_variable_size,...,7950,NaN,0.718930,1366,4860,1.0,7950,7950,Same shape; input is mostly preserved while ne...,[2]
5,task050,50,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,8,1,262,271,same_shape_variable_size,...,1289,NaN,0.922996,365,4740,1.0,1289,1289,Same shape; input is mostly preserved while ne...,[3]
6,task060,60,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,2,1,262,265,same_shape_variable_size,...,4095,NaN,0.792121,686,3300,1.0,4095,4095,Same shape; input is mostly preserved while ne...,[5]
7,task063,63,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,8094,NaN,0.777271,2001,8984,1.0,8094,8094,Same shape; input is mostly preserved while ne...,[3]
8,task090,90,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,4,1,262,267,same_shape_variable_size,...,2984,NaN,0.909534,483,5339,1.0,2984,2984,Same shape; input is mostly preserved while ne...,[6]
9,task102,102,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,4,1,262,267,same_shape_variable_size,...,978,NaN,0.950463,428,8640,1.0,978,978,Same shape; input is mostly preserved while ne...,[2]


In [5]:
# Inspect one scoped task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this subtype.')


task002 examples: 268
first input shape: (6, 6)
first output shape: (6, 6)
first input: [[0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0], [0, 3, 0, 3, 0, 0], [0, 0, 3, 0, 3, 0], [0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0]]
first output: [[0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0], [0, 3, 4, 3, 0, 0], [0, 0, 3, 4, 3, 0], [0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0]]


In [6]:
# nonlocal_1color selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'local_3x3_score',
    'local_3x3_conflicts',
    'added_nonzero_cells',
    'candidate_flags',
]
fill_selection = nonlocal_1color_df[selection_cols].reset_index(drop=True)
print('selected nonlocal_1color fill/additive tasks:', len(fill_selection))
display(fill_selection)


selected nonlocal_1color fill/additive tasks: 41


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,local_3x3_score,local_3x3_conflicts,added_nonzero_cells,candidate_flags
0,task002,medium,5,1,262,same_shape_variable_size,20x20:75;19x19:40;18x18:39;16x16:21;15x15:20,20x20:75;19x19:40;18x18:39;16x16:21;15x15:20,"[0,3]","[0,3,4]",[4],0.919284,1421,9828,adds_new_color_preserves_input|new_output_colors
1,task027,medium,3,1,261,same_shape_variable_size,10x10:265,10x10:265,"[0,1]","[0,1,2]",[2],0.951500,291,1260,adds_new_color_preserves_input|new_output_colors
2,task042,medium,3,1,262,same_shape_variable_size,10x10:266,10x10:266,"[0,3]","[0,3,8]",[8],0.948667,308,1071,adds_new_color_preserves_input|new_output_colors
3,task043,medium,3,1,262,same_shape_variable_size,10x10:266,10x10:266,"[0,5]","[0,2,5]",[2],0.799167,1205,5392,adds_new_color_preserves_input|new_output_colors
4,task047,medium,2,1,262,same_shape_variable_size,9x9:265,9x9:265,"[0,7,8]","[0,2,7,8]",[2],0.718930,1366,7950,adds_new_color_preserves_input|new_output_colors
5,task050,medium,8,1,262,same_shape_variable_size,14x7:6;11x5:6;10x4:5;8x12:5;3x5:5,14x7:6;11x5:6;10x4:5;8x12:5;3x5:5,"[0,8]","[0,3,8]",[3],0.922996,365,1289,adds_new_color_preserves_input|new_output_colors
6,task060,medium,2,1,262,same_shape_variable_size,5x11:265,5x11:265,"[0,1,2,3,4,6,7,8,9]","[0,1,2,3,4,5,6,7,8,9]",[5],0.792121,686,4095,adds_new_color_preserves_input|new_output_colors
7,task063,medium,3,1,262,same_shape_variable_size,14x14:93;10x10:87;12x12:86,14x14:93;10x10:87;12x12:86,"[0,2,8]","[0,2,3,8]",[3],0.777271,2001,8094,adds_new_color_preserves_input|new_output_colors
8,task090,medium,4,1,262,same_shape_variable_size,4x21:13;2x20:11;4x29:11;4x24:10;4x22:10,4x21:13;2x20:11;4x29:11;4x24:10;4x22:10,"[0,1,5]","[0,1,5,6]",[6],0.909534,483,2984,adds_new_color_preserves_input|new_output_colors
9,task102,medium,4,1,262,same_shape_variable_size,12x12:267,12x12:267,"[0,5]","[0,2,5]",[2],0.950463,428,978,adds_new_color_preserves_input|new_output_colors


In [7]:
# Minimal nonlocal_1color v1 trainer: bounded wider exact additive templates plus identity fallback.
# The model preserves the input through a 1x1 identity branch, then adds learned correction logits
# from exact 5x5/7x7 input-patch detectors. A task is accepted only when visible examples have
# zero patch conflicts and the detector count stays under the cap.

CLEAR = 10
ZERO_HOT = -1
NO_CHANGE = -2
MAX_WIDE_TEMPLATE_DETECTORS = 1800
WIDE_TEMPLATE_KERNELS = (5, 7)


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def delta_color(input_color, output_color):
    if output_color == ZERO_HOT or input_color == output_color:
        return NO_CHANGE
    return int(output_color)


def extract_patch_key_k(canvas, row, col, kernel_size):
    pad = kernel_size // 2
    vals = []
    for dr in range(-pad, pad + 1):
        for dc in range(-pad, pad + 1):
            rr = row + dr
            cc = col + dc
            if 0 <= rr < H and 0 <= cc < W:
                vals.append(int(canvas[rr, cc]))
            else:
                vals.append(CLEAR)
    return tuple(vals)


def estimated_detector_static_memory(detector_count):
    # Rough static activation allowance used only for dry-run comparison/reporting.
    return int(detector_count) * H * W * 4


def estimated_wide_exact_params(kernel_size, detector_count):
    # base identity Conv W/B + detector Conv W/B + correction Conv W/B
    return int((CH * CH + CH) + detector_count * (CH * kernel_size * kernel_size + 1 + CH) + CH)


def estimated_competition_cost(info):
    params = info.get('estimated_params') or 10**9
    static_memory = info.get('estimated_static_memory_bytes') or 0
    return int(params + static_memory)


def learn_wide_delta_rules(task, kernel_size, max_detectors=MAX_WIDE_TEMPLATE_DETECTORS):
    patch_to_delta = {}
    conflicts = 0
    total_positions = 0
    changed_positions = 0
    for ex in all_examples(task):
        x = input_canvas(ex['input'])
        y = output_canvas(ex['output'])
        for r in range(H):
            for c in range(W):
                patch = extract_patch_key_k(x, r, c, kernel_size)
                color = delta_color(int(x[r, c]), int(y[r, c]))
                changed_positions += int(color != NO_CHANGE)
                total_positions += 1
                prev = patch_to_delta.get(patch)
                if prev is None:
                    patch_to_delta[patch] = color
                elif prev != color:
                    return None, {
                        'total_positions': total_positions,
                        'changed_positions': changed_positions,
                        'unique_patches': len(patch_to_delta),
                        'conflicts': conflicts + 1,
                    }
    rules = [
        {'template': template, 'delta': int(delta)}
        for template, delta in patch_to_delta.items()
        if int(delta) != NO_CHANGE
    ]
    stats = {
        'total_positions': total_positions,
        'changed_positions': changed_positions,
        'unique_patches': len(patch_to_delta),
        'conflicts': conflicts,
        'template_count': len(rules),
    }
    if len(rules) > max_detectors:
        return None, {**stats, 'reason': 'too many wide exact template detectors'}
    return rules, stats


def fit_wide_exact_additive_rules(task):
    last_info = None
    for kernel_size in WIDE_TEMPLATE_KERNELS:
        rules, stats = learn_wide_delta_rules(task, kernel_size)
        detector_count = int(stats.get('template_count') or 0)
        info = {
            'ok': False,
            'trainer': 'wide_exact_additive_template_cnn',
            'model_version': MODEL_VERSION,
            'kernel_shape': f'[{kernel_size}, {kernel_size}]',
            'pads': f'[{kernel_size // 2}, {kernel_size // 2}, {kernel_size // 2}, {kernel_size // 2}]',
            'max_detectors': MAX_WIDE_TEMPLATE_DETECTORS,
            'template_count': detector_count,
            'rule_count': detector_count,
            'expanded_detector_count': detector_count,
            'estimated_params': estimated_wide_exact_params(kernel_size, detector_count),
            'estimated_static_memory_bytes': estimated_detector_static_memory(detector_count),
            **stats,
        }
        info['estimated_competition_cost'] = estimated_competition_cost(info)
        if rules is None:
            last_info = {**info, 'reason': stats.get('reason') or 'wide exact template conflicts'}
            continue
        if detector_count == 0:
            last_info = {**info, 'reason': 'no added-cell templates'}
            continue
        return {'kind': 'wide_exact_additive', 'kernel_size': int(kernel_size), 'rules': rules}, {**info, 'ok': True, 'reason': None}
    return None, last_info or {
        'ok': False,
        'trainer': 'wide_exact_additive_template_cnn',
        'model_version': MODEL_VERSION,
        'reason': 'no wide exact template rules learned',
    }


def make_wide_exact_additive_model(payload):
    require_onnx()
    kernel_size = int(payload['kernel_size'])
    pad = kernel_size // 2
    rules = payload['rules']
    n_rules = len(rules)
    if n_rules == 0:
        return None

    W_base = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    B_base = np.full((CH,), -0.5, dtype=np.float16)
    for color in range(CH):
        W_base[color, color, 0, 0] = 1.0

    W_det = np.zeros((n_rules, CH, kernel_size, kernel_size), dtype=np.float16)
    B_det = np.zeros((n_rules,), dtype=np.float16)
    center_pos = (kernel_size * kernel_size) // 2
    for detector_idx, rule in enumerate(rules):
        ones = 0
        for idx, expected_color in enumerate(rule['template']):
            kr = idx // kernel_size
            kc = idx % kernel_size
            expected_color = int(expected_color)
            if expected_color == CLEAR:
                W_det[detector_idx, :, kr, kc] = -1.0
            else:
                W_det[detector_idx, :, kr, kc] = -1.0
                W_det[detector_idx, expected_color, kr, kc] = 1.0
                ones += 1
        B_det[detector_idx] = -(ones - 0.5)

    W_delta = np.zeros((CH, n_rules, 1, 1), dtype=np.float16)
    B_delta = np.zeros((CH,), dtype=np.float16)
    for detector_idx, rule in enumerate(rules):
        out_color = int(rule['delta'])
        center_color = int(rule['template'][center_pos])
        if center_color == CLEAR:
            center_color = 0
        W_delta[out_color, detector_idx, 0, 0] = 2.0
        W_delta[center_color, detector_idx, 0, 0] -= 2.0

    initializers = [
        make_initializer('W_base', W_base),
        make_initializer('B_base', B_base),
        make_initializer('W_wide_det', W_det),
        make_initializer('B_wide_det', B_det),
        make_initializer('W_wide_delta', W_delta),
        make_initializer('B_wide_delta', B_delta),
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_wide_det', 'B_wide_det'], ['wide_det_logits'], kernel_shape=[kernel_size, kernel_size], pads=[pad, pad, pad, pad]),
        helper.make_node('Relu', ['wide_det_logits'], ['wide_det_hits']),
        helper.make_node('Conv', ['wide_det_hits', 'W_wide_delta', 'B_wide_delta'], ['wide_delta_logits'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'wide_delta_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)


def train_family_task(task):
    payload, info = fit_wide_exact_additive_rules(task)
    if info.get('ok'):
        try:
            model = make_wide_exact_additive_model(payload)
            if model is not None:
                return model, info
        except Exception as exc:
            return None, {**info, 'ok': False, 'reason': 'wide exact additive export failed', 'wide_export_error': repr(exc)}
    return None, {
        'ok': False,
        'trainer': 'identity_fallback_export',
        'model_version': MODEL_VERSION,
        'reason': 'no bounded visible-fitting wide exact additive model; identity fallback used by build cell',
        'wide_reason': info.get('reason'),
        'wide_kernel': info.get('kernel_shape'),
        'wide_template_count': info.get('template_count'),
        'wide_conflicts': info.get('conflicts'),
    }


In [8]:
# Dry-run nonlocal_1color selector on the selected tasks without saving.
# This reports the exact v1 decision: wide exact additive model or identity fallback.
if 'task_ids' not in globals():
    task_map = load_task_type_map()
    family_df = task_map[task_map.primary_family == FAMILY].copy()
    family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
    nonlocal_1color_df = family_df[
        family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
        & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
        & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
    ].copy().sort_values('task_id').reset_index(drop=True)
    task_ids = nonlocal_1color_df['task_id'].tolist()

dry_rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    payload, info = fit_wide_exact_additive_rules(task)
    selected_trainer = info.get('trainer') if info.get('ok') else 'identity_fallback_export'
    dry_rows.append({
        'task_id': task_id,
        'selected_trainer': selected_trainer,
        'selected_reason': info.get('reason'),
        'wide_ok': bool(info.get('ok')),
        'wide_kernel': info.get('kernel_shape'),
        'wide_templates': info.get('template_count'),
        'wide_conflicts': info.get('conflicts'),
        'estimated_params': info.get('estimated_params'),
        'estimated_static_memory_bytes': info.get('estimated_static_memory_bytes'),
        'estimated_competition_cost': info.get('estimated_competition_cost'),
    })

dry_df = pd.DataFrame(dry_rows)
display(dry_df)
display(dry_df['selected_trainer'].value_counts().rename_axis('selected_trainer').reset_index(name='count'))


,task_id,selected_trainer,selected_reason,wide_ok,wide_kernel,wide_templates,wide_conflicts,estimated_params,estimated_static_memory_bytes,estimated_competition_cost
0,task002,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
1,task027,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
2,task042,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
3,task043,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
4,task047,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
5,task050,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
6,task060,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
7,task063,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
8,task090,identity_fallback_export,wide exact template conflicts,False,"[7, 7]",0,1,120,0,120
9,task102,wide_exact_additive_template_cnn,None,True,"[7, 7]",518,0,259638,1864800,2124438


,selected_trainer,count
0,identity_fallback_export,39
1,wide_exact_additive_template_cnn,2


In [9]:
# Build one model file for every selected nonlocal_1color task.
# The build uses bounded visible-fitting wide exact additive models where available and identity fallback otherwise.
import shutil

if 'task_ids' not in globals():
    task_map = load_task_type_map()
    family_df = task_map[task_map.primary_family == FAMILY].copy()
    family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
    nonlocal_1color_df = family_df[
        family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
        & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
        & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
    ].copy().sort_values('task_id').reset_index(drop=True)
    task_ids = nonlocal_1color_df['task_id'].tolist()

# Clear stale models from earlier runs before creating this scoped zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

print('pre-build task ids:', task_ids)
rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=True,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected nonlocal_1color tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

expected_names = {f'{task_id}.onnx' for task_id in task_ids}
actual_names = {path.name for path in OUT_DIR.glob('task*.onnx')}
print('missing models:', sorted(expected_names - actual_names))
print('extra models:', sorted(actual_names - expected_names))
assert not (expected_names - actual_names), 'missing scoped task models'
assert not (actual_names - expected_names), 'found stale or out-of-scope task models'

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)



pre-build task ids: ['task002', 'task027', 'task042', 'task043', 'task047', 'task050', 'task060', 'task063', 'task090', 'task102', 'task105', 'task119', 'task126', 'task139', 'task162', 'task166', 'task176', 'task200', 'task219', 'task232', 'task246', 'task251', 'task255', 'task265', 'task273', 'task278', 'task299', 'task303', 'task323', 'task335', 'task336', 'task341', 'task348', 'task350', 'task357', 'task367', 'task371', 'task381', 'task387', 'task392', 'task397']


,task_id,saved,path,ok,trainer,model_version,reason,wide_reason,wide_kernel,wide_template_count,...,template_count,rule_count,expanded_detector_count,estimated_params,estimated_static_memory_bytes,total_positions,changed_positions,unique_patches,conflicts,estimated_competition_cost
0,task002,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,task027,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,task042,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,task043,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,task047,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,task050,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,task060,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,task063,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,task090,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.1,no bounded visible-fitting wide exact additive...,wide exact template conflicts,"[7, 7]",0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,task102,True,/kaggle/working/working_submission/fill_enclos...,True,wide_exact_additive_template_cnn,fill-additive-nonlocal-1color-v0.1,None,NaN,NaN,NaN,...,518.0,518.0,518.0,259638.0,1864800.0,240300.0,978.0,25083.0,0.0,2124438.0


selected nonlocal_1color tasks: 41
models saved: 41


,trainer,count
0,identity_fallback_export,39
1,wide_exact_additive_template_cnn,2


missing models: []
extra models: []
family zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'subtype': SUBTYPE,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'task_ids': task_ids,
    'out_dir': str(OUT_DIR),
    'strategy': 'bounded wide exact additive templates, compact additive fallback, identity export fallback',
}
run_manifest


{'family': 'fill_enclosed_regions',
 'subtype': 'nonlocal_1color',
 'model_version': 'fill-additive-nonlocal-1color-v0.1',
 'task_count': 41,
 'task_ids': ['task002',
  'task027',
  'task042',
  'task043',
  'task047',
  'task050',
  'task060',
  'task063',
  'task090',
  'task102',
  'task105',
  'task119',
  'task126',
  'task139',
  'task162',
  'task166',
  'task176',
  'task200',
  'task219',
  'task232',
  'task246',
  'task251',
  'task255',
  'task265',
  'task273',
  'task278',
  'task299',
  'task303',
  'task323',
  'task335',
  'task336',
  'task341',
  'task348',
  'task350',
  'task357',
  'task367',
  'task371',
  'task381',
  'task387',
  'task392',
  'task397'],
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color',
 'strategy': 'bounded wide exact additive templates, compact additive fallback, identity export fallback'}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

pd.DataFrame(validate_rows)

,task_id,right,wrong
0,task002,0,268
1,task027,0,265
2,task042,0,266
3,task043,0,266
4,task047,0,265
5,task050,12,259
6,task060,0,265
7,task063,0,266
8,task090,0,267
9,task102,267,0


In [12]:
# Submission helper.
# Rebuild the zip from this scoped OUT_DIR only.
submission_zip = create_submission_zip(OUT_DIR)
kaggle_submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
import shutil
shutil.copy2(submission_zip, kaggle_submission_zip)
print('scoped zip:', submission_zip)
print('kaggle submission zip:', kaggle_submission_zip)


scoped zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [13]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task002,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,5,0.00,0,1,0.0,0,262,0.000000,0,268,0.00000
1,task027,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,261,0.000000,0,265,0.00000
2,task042,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.00000
3,task043,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.00000
4,task047,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,2,0.00,0,1,0.0,0,262,0.000000,0,265,0.00000
5,task050,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,2,8,0.25,0,1,0.0,10,262,0.038168,12,271,0.04428
6,task060,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,2,0.00,0,1,0.0,0,262,0.000000,0,265,0.00000
7,task063,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.00000
8,task090,fill-additive-nonlocal-1color-v0.1,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,4,0.00,0,1,0.0,0,262,0.000000,0,267,0.00000
9,task102,fill-additive-nonlocal-1color-v0.1,519942,259638,7,"{""Add"": 1, ""Cast"": 2, ""Conv"": 3, ""Relu"": 1}",1936800,2456076,4,4,1.00,1,1,1.0,262,262,1.000000,267,267,1.00000


In [14]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.1_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.1_manifest.json
